In [1]:
import json
import re

def extract_answer_from_response(response, dataset):
    """
    Extract answer from response based on dataset type
    """
    if not response:
        return None
    
    # For LogiQA: Look for A, B, C, D inside <think> tags
    if dataset.lower() == 'logiqa':
        think_match = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
        if think_match:
            thinking = think_match.group(1)
            answer_match = re.search(r'(?:answer|option|choice)\s*(?:is|:|=)?\s*([A-D])\b', thinking, re.IGNORECASE)
            if answer_match:
                return answer_match.group(1).upper()
            
            # Pattern 2: Just the letter near the end
            letters = re.findall(r'\b([A-D])\b', thinking)
            if letters:
                return letters[-1].upper() 
    
    # For math datasets: Look for #### format
    answer_match = re.search(r'####\s*(.+?)(?:\n|</think>|$)', response)
    if answer_match:
        return answer_match.group(1).strip()
    
    # Look for \boxed{} format
    boxed_match = re.search(r'\\boxed\{([^}]+)\}', response)
    if boxed_match:
        return boxed_match.group(1).strip()
    
    return None

def convert_to_grpo_format(input_file, output_file):
    """
    Convert to GRPO format - extract answers from responses, then remove responses
    """
    
    with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
        stats = {
            'total': 0,
            'answer_from_response': 0,
            'answer_from_label': 0,
            'no_answer': 0,
            'by_dataset': {}
        }
        
        for line in f_in:
            item = json.loads(line)
            stats['total'] += 1
            
            # Extract components
            prompt = item.get('prompt', '')
            instruction = item.get('instruction', '')
            response = item.get('response', '')
            mode = item.get('mode', 'low')
            label = item.get('label', '')
            dataset = item.get('dataset', 'unknown')
            
            # Track by dataset
            if dataset not in stats['by_dataset']:
                stats['by_dataset'][dataset] = {'total': 0, 'from_response': 0, 'from_label': 0}
            stats['by_dataset'][dataset]['total'] += 1
            
            # Build query with instruction
            query = f"{instruction}\n\n{prompt}" if instruction else prompt
            
            # Try to extract answer from response first
            extracted_answer = extract_answer_from_response(response, dataset)
            
            if extracted_answer:
                stats['answer_from_response'] += 1
                stats['by_dataset'][dataset]['from_response'] += 1
            else:
                # For LogiQA labels
                if dataset.lower() == 'logiqa':
                    letter_match = re.search(r'\b([A-D])\b', label)
                    if letter_match:
                        extracted_answer = letter_match.group(1).upper()
                
                # For math labels with ####
                if not extracted_answer:
                    answer_match = re.search(r'####\s*(.+?)(?:\n|$)', label)
                    if answer_match:
                        extracted_answer = answer_match.group(1).strip()
                
                # Fallback: try to find any number
                if not extracted_answer:
                    nums = re.findall(r'-?\d+\.?\d*', label)
                    extracted_answer = nums[-1] if nums else label.strip()
                
                if extracted_answer:
                    stats['answer_from_label'] += 1
                    stats['by_dataset'][dataset]['from_label'] += 1
                else:
                    stats['no_answer'] += 1
            
            # GRPO format - NO RESPONSE, model generates it
            grpo_item = {
                'query': query,
                'mode': mode,
                'answer': extracted_answer if extracted_answer else 'unknown',
            }
            
            f_out.write(json.dumps(grpo_item) + '\n')
        
        # Print statistics
        print(f"\n✓ Conversion complete!")
        print(f"  Total examples: {stats['total']}")
        print(f"  Answer from response: {stats['answer_from_response']} ({100*stats['answer_from_response']/stats['total']:.1f}%)")
        print(f"  Answer from label: {stats['answer_from_label']} ({100*stats['answer_from_label']/stats['total']:.1f}%)")
        print(f"  No answer found: {stats['no_answer']} ({100*stats['no_answer']/stats['total']:.1f}%)")
        
        print(f"\n📊 By dataset:")
        for dataset, counts in stats['by_dataset'].items():
            print(f"\n  {dataset}:")
            print(f"    Total: {counts['total']}")
            print(f"    From response: {counts['from_response']}")
            print(f"    From label: {counts['from_label']}")

# Convert
convert_to_grpo_format(
    '/workspace/LLaMA-OSS/LLaMA-Factory/data/combined_grpo_train.jsonl',
    '/workspace/LLaMA-OSS/train_grpo_formatted.jsonl'
)

# Show examples by dataset
print("\n" + "="*60)
print("Sample outputs by dataset:")
print("="*60)

with open('/workspace/LLaMA-OSS/train_grpo_formatted.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]

# Get original data to show dataset
with open('/workspace/LLaMA-OSS/LLaMA-Factory/data/combined_grpo_train.jsonl', 'r') as f:
    original = [json.loads(line) for line in f]

# Show one example per dataset
seen_datasets = set()
for orig, converted in zip(original, data):
    dataset = orig.get('dataset', 'unknown')
    if dataset not in seen_datasets:
        seen_datasets.add(dataset)
        print(f"\n--- {dataset.upper()} Example ---")
        print(f"Query: {converted['query'][:80]}...")
        print(f"Mode: {converted['mode']}")
        print(f"Answer: {converted['answer']}")
        print(f"(Extracted from: {orig.get('response', '')[:100]}...)")


✓ Conversion complete!
  Total examples: 26106
  Answer from response: 26094 (100.0%)
  Answer from label: 12 (0.0%)
  No answer found: 0 (0.0%)

📊 By dataset:

  gsm8k:
    Total: 9132
    From response: 9132
    From label: 0

  logiqa:
    Total: 5298
    From response: 5298
    From label: 0

  compmath:
    Total: 11676
    From response: 11664
    From label: 12

Sample outputs by dataset:

--- GSM8K Example ---
Query: Respond concisely with minimal reasoning.

Natalia sold clips to 48 of her frien...
Mode: low
Answer: 72
(Extracted from: <think>Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips alt...)

--- LOGIQA Example ---
Query: Respond concisely with minimal reasoning.

Context: Continuous exposure to indoo...
Mode: low
Answer: A
(Extracted from: <think>A</think>...)

--- COMPMATH Example ---
Query: Respond concisely with minimal reasoning.

Evaluate $\left\lceil3\left(6-\frac12...
Mode: low
Answer: 17
(Extracted from: <think>Firstly,

In [2]:
import json
import numpy as np
from collections import defaultdict
import re

def analyze_thinking_lengths(files):
    """
    Analyze thinking lengths for each file and mode
    """
    
    results = {}
    
    for file_path in files:
        print(f"\n{'='*80}")
        print(f"Analyzing: {file_path}")
        print(f"{'='*80}")
        
        # Read data
        data = []
        try:
            with open(file_path, 'r') as f:
                for line in f:
                    data.append(json.loads(line))
        except FileNotFoundError:
            print(f"❌ File not found: {file_path}")
            continue
        
        print(f"Total examples: {len(data)}")
        
        # Group by mode and extract thinking lengths
        mode_lengths = defaultdict(list)
        
        for item in data:
            mode = item.get('mode', 'unknown').lower()
            response = item.get('response', '')
            
            # Extract thinking content
            match = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
            if match:
                thinking = match.group(1).strip()
                # Approximate token count (characters / 4)
                length = len(thinking) // 4
                mode_lengths[mode].append(length)
        
        # Calculate statistics for each mode
        file_results = {}
        for mode in ['low', 'medium', 'high']:
            if mode in mode_lengths:
                lengths = mode_lengths[mode]
                mean = np.mean(lengths)
                std = np.std(lengths)
                
                file_results[mode] = {
                    'count': len(lengths),
                    'mean': mean,
                    'std': std,
                    'min': min(lengths),
                    'max': max(lengths),
                    'median': np.median(lengths)
                }
                
                print(f"\n{mode.upper()}:")
                print(f"  Count: {len(lengths)}")
                print(f"  Mean: {mean:.2f} tokens")
                print(f"  Std: {std:.2f} tokens")
                print(f"  Min: {min(lengths)} tokens")
                print(f"  Max: {max(lengths)} tokens")
                print(f"  Median: {np.median(lengths):.0f} tokens")
        
        results[file_path] = file_results
    
    # Summary table
    print("\n" + "="*80)
    print("SUMMARY TABLE")
    print("="*80)
    print(f"\n{'File':<40} {'Mode':<10} {'Count':<8} {'Mean':<10} {'Std':<10}")
    print("-" * 80)
    
    for file_path, modes in results.items():
        filename = file_path.split('/')[-1]
        for mode in ['low', 'medium', 'high']:
            if mode in modes:
                stats = modes[mode]
                print(f"{filename:<40} {mode:<10} {stats['count']:<8} {stats['mean']:<10.1f} {stats['std']:<10.1f}")
    
    # Code output
    print("\n" + "="*80)
    print("FOR REWARD MODEL CODE:")
    print("="*80)
    
    for file_path, modes in results.items():
        filename = file_path.split('/')[-1]
        print(f"\n# {filename}")
        print("self.mode_params = {")
        for mode in ['low', 'medium', 'high']:
            if mode in modes:
                mean = int(modes[mode]['mean'])
                std = int(modes[mode]['std'])
                print(f"    '{mode}': {{'mean': {mean}, 'std': {std}}},")
        print("}")

# Analyze all three files
files = [
    '/workspace/LLaMA-OSS/LLaMA-Factory/data/combined_sft_train_high.jsonl',
    '/workspace/LLaMA-OSS/LLaMA-Factory/data/combined_sft_train_low.jsonl',
    '/workspace/LLaMA-OSS/LLaMA-Factory/data/combined_sft_train_medium.jsonl'
]

analyze_thinking_lengths(files)


Analyzing: /workspace/LLaMA-OSS/LLaMA-Factory/data/combined_sft_train_high.jsonl
Total examples: 5641



HIGH:
  Count: 5641
  Mean: 1514.27 tokens
  Std: 452.97 tokens
  Min: 424 tokens
  Max: 2880 tokens
  Median: 1470 tokens

Analyzing: /workspace/LLaMA-OSS/LLaMA-Factory/data/combined_sft_train_low.jsonl
Total examples: 10388

LOW:
  Count: 10388
  Mean: 104.75 tokens
  Std: 53.86 tokens
  Min: 24 tokens
  Max: 337 tokens
  Median: 94 tokens

Analyzing: /workspace/LLaMA-OSS/LLaMA-Factory/data/combined_sft_train_medium.jsonl
Total examples: 9485

MEDIUM:
  Count: 9485
  Mean: 501.84 tokens
  Std: 236.30 tokens
  Min: 104 tokens
  Max: 1356 tokens
  Median: 455 tokens

SUMMARY TABLE

File                                     Mode       Count    Mean       Std       
--------------------------------------------------------------------------------
combined_sft_train_high.jsonl            high       5641     1514.3     453.0     
combined_sft_train_low.jsonl             low        10388    104.7      53.9      
combined_sft_train_medium.jsonl          medium     9485     501.8      236.3   